In [19]:
# ==============================
# 🔧 INSTALAÇÃO
# ==============================
!pip install -q sentence-transformers bert-score transformers evaluate
!pip install -q evaluate
!pip install -q git+https://github.com/google-research/bleurt.git

# ==============================
# 📌 IMPORTS
# ==============================
from sentence_transformers import SentenceTransformer, util
from bert_score import score as bertscore
from transformers import pipeline
import evaluate
import re
import torch

# ==============================
# ✍️ INPUT
# ==============================
GABARITO = "O réu deve ser absolvido pela lei 2010."
RESPOSTA = "O réu deve ser absolvido."
LEGISLACAO = "2010"

# ==============================
# 🧠 MODELOS
# ==============================
#Modelo do domínio jurídico português
sbert_model = SentenceTransformer('stjiris/bert-large-portuguese-cased-legal-mlm-sts-v1.0')

#Modelo NLI multilingual - tem o português
device = 0 if torch.cuda.is_available() else -1
nli_model = pipeline(
    "text-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
    device=device
)

#Modelo BLEURT multilingual - tem o português
bleurt_model = evaluate.load("bleurt", "BLEURT-20")

# ==============================
# 🔍 1. BERTSCORE
# ==============================
P, R, F1 = bertscore([RESPOSTA], [GABARITO], lang="pt", verbose=False)

bertscore_result = {
    "precision": P.item(),
    "recall": R.item(),
    "f1": F1.item()
}

# ==============================
# 🔍 2. SBERTSCORE
# ==============================
def segmentar_sentencas(texto):
    return [s.strip() for s in texto.split('.') if s.strip()]

resp_sents = segmentar_sentencas(RESPOSTA)
gab_sents = segmentar_sentencas(GABARITO)

emb_resp = sbert_model.encode(resp_sents, convert_to_tensor=True)
emb_gab = sbert_model.encode(gab_sents, convert_to_tensor=True)

cos_sim = util.cos_sim(emb_resp, emb_gab)

precision = cos_sim.max(dim=1).values.mean().item()
recall = cos_sim.max(dim=0).values.mean().item()
f1 = 2 * (precision * recall) / (precision + recall)

sbertscore_result = {
    "precision": precision,
    "recall": recall,
    "f1": f1
}

# ==============================
# 🔍 3. BLEURT
# ==============================
def calcular_bleurt(resposta, gabarito):
    result = bleurt_model.compute(
        predictions=[resposta],
        references=[gabarito]
    )
    score = result["scores"][0]
    return {"score": score}

bleurt_result = calcular_bleurt(RESPOSTA, GABARITO)

# ==============================
# 🔍 4. MATCH LEGISLAÇÃO (ROBUSTO)
# ==============================
def normalizar_numero(num):
    """Remove pontos e zeros à esquerda"""
    return re.sub(r'\D', '', num).lstrip('0')

def extrair_referencias_legais(texto):
    """
    Captura padrões como:
    - Lei 8.112
    - Lei nº 8.112/90
    - 8112
    - 8.112
    """
    padroes = [
        r'lei\s*(?:n[ºo]\s*)?(\d{1,5}(?:\.\d{3})*(?:/\d{2,4})?)',
        r'\b(\d{1,5}(?:\.\d{3})*(?:/\d{2,4})?)\b'
    ]

    encontrados = []

    for padrao in padroes:
        matches = re.findall(padrao, texto.lower())
        encontrados.extend(matches)

    return list(dict.fromkeys(normalizar_numero(n) for n in encontrados))

def verificar_legislacao(resposta, legislacao_esperada):
    if not legislacao_esperada:
        return {
            "esperada": None,
            "encontrados": [],
            "match": None
        }

    esperado_norm = normalizar_numero(legislacao_esperada)
    encontrados = extrair_referencias_legais(resposta)

    match = esperado_norm in encontrados

    return {
        "esperada": legislacao_esperada,
        "esperada_norm": esperado_norm,
        "encontrados": encontrados,
        "match": match
    }

legislacao_result = verificar_legislacao(RESPOSTA, LEGISLACAO)

# ==============================
# 🔍 5. NLI
# ==============================
def calcular_nli(resposta, gabarito):
    result = nli_model(f"{gabarito} </s> {resposta}")

    label = result[0]['label'].upper()
    score = result[0]['score']

    if label == "ENTAILMENT":
        nli_score = score
    elif label == "NEUTRAL":
        nli_score = 0.5 * score
    elif label == "CONTRADICTION":
        nli_score = 1 - score

    return {
        "label": label,
        "score_bruto": score,
        "score_convertido": nli_score
    }

nli_result = calcular_nli(RESPOSTA, GABARITO)

# ==============================
# 📊 RESULTADOS
# ==============================
print("\n==============================")
print("📊 RESULTADOS")
print("==============================\n")

print("🔹 BERTScore:")
print(bertscore_result)

print("\n🔹 SBERTScore:")
print(sbertscore_result)

print("\n🔹 BLEURT:")
print(bleurt_result)

print("\n🔹 Legislação:")
print(legislacao_result)

print("\n🔹 NLI:")
print(nli_result)

# ==============================
# 💬 6. EXPLICAÇÃO
# ==============================
def gerar_explicacao(bert_f1, sbert_f1, bleurt_score, legislacao_result, nli_result ):
    explicacao = []
    # --- BERTScore ---
    if bert_f1 >= 0.90:
        explicacao.append(f"✅ Similaridade semântica muito alta (BERTScore F1: {bert_f1:.2f}).")
    elif bert_f1 >= 0.75:
        explicacao.append(f"⚠️ Similaridade semântica boa (BERTScore F1: {bert_f1:.2f}).")
    else:
        explicacao.append(f"❌ Baixa similaridade semântica em nível de tokens (BERTScore F1: {bert_f1:.2f}).")

    # --- SBERT ---
    if sbert_f1 >= 0.85:
        explicacao.append(f"✅ Alta similaridade semântica com o gabarito (SBERTScore F1: {sbert_f1:.2f}).")
    elif sbert_f1 >= 0.65:
        explicacao.append(f"⚠️ Similaridade semântica moderada com o gabarito (SBERTScore F1: {sbert_f1:.2f}).")
    else:
        explicacao.append(f"❌ Baixa similaridade semântica com o gabarito (SBERTScore F1: {sbert_f1:.2f}).")

    # --- BLEURT ---
    if bleurt_score >= 0.5:
        explicacao.append(f"✅ Alta qualidade de geração (BLEURT: {bleurt_score:.2f}).")
    elif bleurt_score >= 0.0:
        explicacao.append(f"⚠️ Qualidade moderada de geração (BLEURT: {bleurt_score:.2f}).")
    else:
        explicacao.append(f"❌ Baixa qualidade de geração (BLEURT: {bleurt_score:.2f}).")

    # --- LEGISLAÇÃO ---
    if legislacao_result["match"] is None:
        explicacao.append("ℹ️ Nenhuma legislação foi exigida para esta questão.")
    elif legislacao_result["match"]:
        explicacao.append(f"✅ Legislação correta mencionada (Lei {legislacao_result['esperada']} identificada na resposta).")
    else:
        encontrados = legislacao_result.get("encontrados", [])
        if encontrados:
            explicacao.append(f"❌ Legislação incorreta: era esperada a Lei {legislacao_result['esperada']}, mas foram encontradas referências a: {', '.join(encontrados)}. Penalização.")
        else:
            explicacao.append(f"❌ Nenhuma legislação foi mencionada na resposta (esperada: Lei {legislacao_result['esperada']}). Penalização.")

    # --- NLI ---
    label = nli_result["label"]
    nli_score = nli_result["score_convertido"]

    if label == "CONTRADICTION":
        explicacao.append(f"❌ A resposta contradiz o gabarito (NLI: contradição com score {nli_result['score_bruto']:.2f}). Penalização.")
    elif label == "ENTAILMENT":
        explicacao.append(f"✅ A resposta está alinhada com o gabarito (NLI: implicação com score {nli_result['score_bruto']:.2f}).")
    elif label == "NEUTRAL":
        explicacao.append(f"⚠️ A resposta é parcialmente compatível com o gabarito (NLI: neutro com score {nli_result['score_bruto']:.2f}).")

    return "\n".join(explicacao)


explicacao = gerar_explicacao(
    bert_f1=bertscore_result["f1"],
    sbert_f1=sbertscore_result["f1"],
    bleurt_score=bleurt_result["score"],
    legislacao_result=legislacao_result,
    nli_result=nli_result
)

print("\n🔹 Explicação:")
print(explicacao)

  Preparing metadata (setup.py) ... done


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: stjiris/bert-large-portuguese-cased-legal-mlm-sts-v1.0
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing checksums: 100%|##########| 1/1 [00:12<00:00, 12.84s/it]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📊 RESULTADOS

🔹 BERTScore:
{'precision': 0.9105892181396484, 'recall': 0.8454341292381287, 'f1': 0.876802921295166}

🔹 SBERTScore:
{'precision': 0.7294001579284668, 'recall': 0.7294001579284668, 'f1': 0.7294001579284668}

🔹 BLEURT:
{'score': 0.7527565360069275}

🔹 Legislação:
{'esperada': '2010', 'esperada_norm': '2010', 'encontrados': [], 'match': False}

🔹 NLI:
{'label': 'ENTAILMENT', 'score_bruto': 0.9985716342926025, 'score_convertido': 0.9985716342926025}

🔹 Explicação:
⚠️ Similaridade semântica boa (BERTScore F1: 0.88).
⚠️ Similaridade semântica moderada com o gabarito (SBERTScore F1: 0.73).
✅ Alta qualidade de geração (BLEURT: 0.75).
❌ Nenhuma legislação foi mencionada na resposta (esperada: Lei 2010). Penalização.
✅ A resposta está alinhada com o gabarito (NLI: implicação com score 1.00).
